# 开源增强边缘AI v1.1 · 只读体验
参数通过输入框设置，不需要修改代码单元。未学习配置会拒答；计划预览不执行硬件。

In [ ]:
from pathlib import Path
import sys, json
ROOT=Path.cwd().resolve()
assert (ROOT/'oss_cli.py').is_file(), '请在软件包目录打开本Notebook'
sys.path[:0]=[str(ROOT),str(ROOT/'backend'),str(ROOT/'oss')]
from edge_package import verify
print('固定文件及Notebook代码检查:', verify(notebooks=True))
from portable import load_default, from_condition, STRATEGY
MODEL=load_default()
print('OSS模型:',MODEL.model_sha256)
print('只读软件体验；不会加载Overlay、创建会话或发送START。')


In [ ]:
from custom_ai_core import query
def number(label, default):
    text=input(label+' ['+str(default)+']: ').strip()
    return default if not text else int(text)
d=number('B参数d',3); s=number('B参数s',-1)
byte=number('毛刺字节',20); bit=number('位号',0)
phase=number('相位',15); width=number('宽度',5)
mode=input('模式 TYPE3A_SINGLE / TYPE3C_CONTIGUOUS / DISABLED [TYPE3A_SINGLE]: ').strip() or 'TYPE3A_SINGLE'
count=number('毛刺数量',0 if mode=='DISABLED' else 1)
target=input('目标 [B_frame_correct]: ').strip() or 'B_frame_correct'
QUERY=query(d=d,s=s,byte=byte,bit=bit,phase=phase,width=width,count=count,mode=mode,target=target)
print(json.dumps(MODEL.predict(QUERY),ensure_ascii=False,indent=2))
print('未学习的配置将拒答；预测不是实测标签。')


In [ ]:
from custom_plan import compile_plan
path=input('请求JSON文件路径（留空则预览当前单点，含CONFIG/参照共5条）: ').strip()
if path:
    request=json.loads(Path(path).read_bytes())
else:
    request=dict(receiver_B=QUERY['receiver_B'],mode=QUERY['mode'],
        fault={k:QUERY['fault'][k] for k in ('byte','bit','count')},
        scan=dict(points=[{k:QUERY['fault'][k] for k in ('phase','width')}]),
        repeats=3,max_records=5,target=QUERY['target'])
request['edge_ai']=dict(strategy=STRATEGY,model_file_sha256=MODEL.file_sha256,max_extra_groups=1,allow_rule_fallback=True)
PREVIEW=compile_plan(request)
print('状态:',PREVIEW['status'],'错误:',PREVIEW['errors'])
if PREVIEW['plan']:
    plan=PREVIEW['plan']
    print('总上限:',plan['max_records'],'计划SHA:',plan['sha256'])
    for condition in plan['conditions']:
        print(condition['condition_id'],MODEL.predict(from_condition(condition,plan['target'])))
print('仅计划预览，没有硬件授权，也没有执行任何试次。')


In [ ]:
from IPython.display import HTML, display
display(HTML((ROOT/'oss/report.html').read_text(encoding='utf-8')))
